---
title: Глава 7. Генерация, обработка и преобразование данных
subtitle: SQL Lab in JupyterLab
# license: CC-BY-4.0
github: https://github.com/magus1968/learning-sql
subject: Technical Portfolio
venue: GitHub & GitVerse Pages
# abstract: |
#   В последнем запросе главы, в разделе _Этот таинственный null_, увидим причину, по которой для усечения строк в дальнейшем будет использоваться Pandas.
authors:
  - name: Alex Smirnov
    email: a@smirnovs.pro
    corresponding: true
    affiliations: Data & BI Analyst
date: 2026-08-25
abbreviations:
    MyST: Markedly Structured Text
    Jupyter Book: Build static Web-books
    JupySQL: Run & highlight SQL in Jupyter
---

In [2]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL


connection_url = URL.create(
    drivername="mysql+pymysql",
    host="localhost",
    port=3306,
    database="sakila",
    username="root",
    password="*UHB5rdx",
)

engine = create_engine(connection_url)

%load_ext sql

%config SqlMagic.displaylimit = 20
%config SqlMagic.displaycon = False
# %config SqlMagic.feedback = False

%sql engine

print("SQLAlchemy - подключение создано")
print("JupySQL - успешно подключен через SQLAlchemy Engine!")

SQLAlchemy - подключение создано
JupySQL - успешно подключен через SQLAlchemy Engine!


## Работа со строковыми данными

При работе со строковыми данными используется один из символьных типов данных: `char`, `varchar`, `text` (tinytext, text, mediumtext, longtext).

In [2]:
%%sql
CREATE TABLE string_tbl
    (char_fld CHAR(30),
    vchar_fld VARCHAR(30),
    text_fld TEXT
);

++
||
++
++

### Генерация строк

Самый простой способ заполнить символьный столбец – заключить строку в кавычки.

In [26]:
%%sql
INSERT INTO string_tbl (char_fld, vchar_fld, text_fld)
VALUES ('This is char data',
    'This is varchar data',
    'This is text data');

1 rows affected.

++
||
++
++

In [27]:
%%sql
SELECT * FROM string_tbl;

1 rows affected.

char_fld,vchar_fld,text_fld
This is char data,This is varchar data,This is text data


:::{warning}

При вставке строковых данных: если длина строки превышает максимальный размер для символьного столбца, сервер сгенерирует **исключение** (exception) – сигнал о возникновении аварийной ситуации, которая нарушает нормальный ход выполнения программы.
:::

Попытаемся заменить столбец vchar_fld строкой в 38 символов:

In [28]:
%%sql
UPDATE string_tbl
SET vchar_fld = 'This is a piece of extremely long data';

RuntimeError: (pymysql.err.DataError) (1406, "Data too long for column 'vchar_fld' at row 1")
[SQL: UPDATE string_tbl
SET vchar_fld = 'This is a piece of extremely long data';]
(Background on this error at: https://sqlalche.me/e/20/9h9h)


Получили исключение; данные остались не тронутыми:

In [29]:
%%sql
SELECT * FROM string_tbl;

1 rows affected.

char_fld,vchar_fld,text_fld
This is char data,This is varchar data,This is text data


Проверим в каком режиме работаем:

In [30]:
%%sql
SELECT @@session.sql_mode;

1 rows affected.

@@session.sql_mode
"ONLY_FULL_GROUP_BY,STRICT_TRANS_TABLES,NO_ZERO_IN_DATE,NO_ZERO_DATE,ERROR_FOR_DIVISION_BY_ZERO,NO_ENGINE_SUBSTITUTION"


Если предпочтем _**механизм усечения строк**_ вместо генерации исключений нужно выбрать режим ANSI:

In [31]:
%%sql
SET sql_mode = 'ansi';

++
||
++
++

In [32]:
%%sql
SELECT @@session.sql_mode;

1 rows affected.

@@session.sql_mode
"REAL_AS_FLOAT,PIPES_AS_CONCAT,ANSI_QUOTES,IGNORE_SPACE,ONLY_FULL_GROUP_BY,ANSI"


Вызвав инструкцию `UPDATE` повторно, обнаружим что сервер выполнит ее без ошибок и обновит столбец:

In [33]:
%%sql
UPDATE string_tbl
SET vchar_fld = 'This is a piece of extremely long data';

1 rows affected.

++
||
++
++

:::{attention} Однако внутри сессии будет зафиксировано скрытое предупреждение
Чтобы принудительно запросить у сервера текст предупреждения, необходимо следующим шагом выполнить команду `SHOW WARNINGS`, чтобы вывести диагностический результат последней выполненной операции.

Получим предупреждение о принудительном усечении текста под максимальный размер столбца: *В первой строке таблицы (at row 1) вы попытались записать в столбец vchar_fld слишком длинный текст. База данных не стала аварийно прерывать работу, но физически отрезала (truncated) лишний хвост строки, который не поместился в лимит столбца, и сохранила только разрешенную часть*.

In [34]:
%%sql
SHOW WARNINGS;

1 rows affected.

Level,Code,Message
Warning,1265,Data truncated for column 'vchar_fld' at row 1


Выполнив выборку столбца *vchar_fld*, увидим что строка действительно была усечена:

In [35]:
%%sql
SELECT vchar_fld
FROM string_tbl;

1 rows affected.

vchar_fld
This is a piece of extremely l


Чтобы вернуть настройки обратно в **безопасное состояние** с включенным **строгим контролем данных**, сбрасываем их в значение по умолчанию:

In [36]:
%%sql
SET sql_mode = DEFAULT;

++
||
++
++

In [37]:
%%sql
SELECT @@session.sql_mode;

1 rows affected.

@@session.sql_mode
"ONLY_FULL_GROUP_BY,STRICT_TRANS_TABLES,NO_ZERO_IN_DATE,NO_ZERO_DATE,ERROR_FOR_DIVISION_BY_ZERO,NO_ENGINE_SUBSTITUTION"


:::{important} Лучший способ избежать исключений или усечения строки

При работе со столбцами `varchar` установить для верхнего порога достаточно высокое значение, которого хватит для обработки самых длинных строк, помещаемых в столбец.

Сервер выделяет для хранения такой строки только необходимое место, поэтому установка высокого предела для столбцов `varchar` не является расточительной.
:::

---

#### Включение одинарных кавычек

Поскольку строки указываются одинарными кавычками, следует обратить особое внимание на строки, содержащие внутри одинарные кавычки или апострофы.

In [38]:
%%sql
UPDATE string_tbl
SET text_fld = 'This string dosn't work';

RuntimeError: If using snippets, you may pass the --with argument explicitly.
For more details please refer: https://jupysql.ploomber.io/en/latest/compose.html#with-argument


Original error message from DB driver:
(pymysql.err.ProgrammingError) (1064, "You have an error in your SQL syntax; check the manual that corresponds to your MySQL server version for the right syntax to use near 't work'' at line 2")
[SQL: UPDATE string_tbl
SET text_fld = 'This string dosn't work';]
(Background on this error at: https://sqlalche.me/e/20/f405)



Чтобы серевер игнорировал апостроф, необходимо добавить перед ним управляющий символ `'` или символ обратной косой черты `\`

In [39]:
%%sql
UPDATE string_tbl
SET text_fld = 'This string didn''t work, but it does now';

1 rows affected.

++
||
++
++

In [40]:
%%sql
UPDATE string_tbl
SET text_fld = 'This string didn\'t work, but it does now';

1 rows affected.

++
||
++
++

In [41]:
%%sql
SELECT text_fld
FROM string_tbl;

1 rows affected.

text_fld
"This string didn't work, but it does now"


Однако, если извлекаете строку для добавления в файл (который будет читать другая программа), то может потребоваться включение управляющего символа как части извлеченной строки.

:::{attention} Функция `quote()`
:class: simple
:icon: false
Заключает всю строку в кавычки и добавляет управляющие символы к любым одинарным кавычкам или апострофам внутри строки.
:::

In [42]:
%%sql
SELECT QUOTE(text_fld)
FROM string_tbl;

1 rows affected.

QUOTE(text_fld)
"'This string didn\'t work, but it does now'"


При получении данных для экспорта целесообразно использовать функцию `quote()` для всех символьных столбцов не генерируемых системой – которые пользователи заполняют вручную на обычном языке (комментарии, описания, адреса, отзывы). В них может встретиться любой неожиданный символ.

:::{hint} Алан Бьюли дает отличный совет для реальной работы

Eсли выгружаете базу данных в текстовый файл для передачи аналитикам или в другую систему, всегда пропускайте текстовые комментарии через `quote()`. Это гарантирует, что файл откроется в Excel идеально ровно.
:::

---

#### Включение специальных символов

Если приложение является многонациональным, в нем, вероятно, будут встречаться строки, содержащие символы, которых нет на клавиатуре и может потребоваться включить такие символы, например, с диакритическими знаками **é** или **ö**.

:::{attention} Функция `char()`
:class: simple
:icon: false
Позволяет создавать строки, содержащие любые из 255 символов в наборе символов ASCII.
:::

In [5]:
%%sql
SELECT 'abcdefg', CHAR(97, 98, 99, 100, 101, 102, 103);

1 rows affected.

abcdefg,"CHAR(97, 98, 99, 100, 101, 102, 103)"
abcdefg,b'abcdefg'


:::{note} Разница в выводе MySQL CLI vs Jupyter

Функция `CHAR()` берет числа (ASCII-коды) и возвращает соответствующую им последовательность байт. По умолчанию в MySQL результат функции `CHAR()` возвращается как бинарная строка (тип данных `BINARY` или `BLOB`), а не как обычный читаемый текст (`VARCHAR`).

Алан Бьюли выполняет запросы в стандартной консоли MySQL CLI (Command Line Client), которая автоматически преобразует бинарные байты в читаемые символы на экране. Она не пытается показать типы данных Python, поэтому у автора в книге вывелся чистый текст `abcdefg`.

Когда Jupyter через Python-драйвер делает запрос к базе данных, он получает от MySQL сырой бинарный поток байт. Современные библиотеки Python (особенно при работе через JupySQL) честно предупреждают: _Внимание, база данных вернула этот текст в виде чистых байт, а не текстовой строки_ – и добавляют маркер b (binary) `b'abcdefg'`.
:::

:::{hint} Чтобы в Jupyter получить чистый текст 

Чтобы MySQL принудительно вернул результат как обычную текстовую строку (а не бинарную), можно явно указать кодировку с помощью ключевого слова `USING` внутри функции `CHAR(97, 98, 99 USING utf8mb4)`
:::

In [6]:
%%sql
SELECT 'abcdefg', CHAR(97, 98, 99, 100, 101, 102, 103 USING utf8mb4);

1 rows affected.

abcdefg,"CHAR(97, 98, 99, 100, 101, 102, 103 USING utf8mb4)"
abcdefg,abcdefg


:::{hint} Чтобы Jupyter вывел те же буквы расширенной латиницы

Что и в оригинале у Алана Бьюли `Çüéâäàåçêë`, нужно использовать кодировку `binary` для функции `CHAR()` и затем перекодировать её в `cp850`.

В оригинальной консоли MySQL у автора использовалась старая DOS-кодировка Western European (`cp850`). Именно в ней числа 128–137 будут теми же буквами что и у автора.
:::

In [26]:
%%sql
SELECT CONVERT(
  CHAR(128, 129, 130, 131, 132, 133, 134, 135, 136, 137 USING binary) 
  USING cp850
) decoded_result;

1 rows affected.

decoded_result
Çüéâäàåçêë


- `CHAR(... USING binary)` заставляет MySQL сгенерировать чистые, неиспорченные байты
- `CONVERT(... USING cp850)` принудительно расшифровывает их по старой западной DOS-таблице автора книги, превращая в правильные буквы Çüéâäàåçêë.
- Драйвер Python видит этот готовый текст и выводит его в Jupyter идеально чисто, без маркера `b` и без кракозябр.

In [30]:
%%sql
SELECT CONVERT(
    CHAR(138, 139, 140, 141, 142, 143, 144, 145, 146, 147 USING binary)
    USING cp850
) decoded_result;

1 rows affected.

decoded_result
èïîìÄÅÉæÆô


In [31]:
%%sql
SELECT CONVERT(
    CHAR(148, 149, 150, 151, 152, 153, 154, 155, 156, 157 USING binary)
    USING cp850
) decoded_result;

1 rows affected.

decoded_result
öòûùÿÖÜø£Ø


In [32]:
%%sql
SELECT CONVERT(
    CHAR(158, 159, 160, 161, 162, 163, 164, 165 USING binary)
    USING cp850
) decoded_result;

1 rows affected.

decoded_result
×ƒáíóúñÑ


Используем функцию `concat()` для соединения отдельных строк, одни из которых просто введем с клавиатуры, а другие сгенерируем с помощью `char()`:

In [37]:
%%sql
SELECT CONCAT(
  'danke sch', 
  CONVERT(CHAR(148 USING binary) USING cp850), 
  'n'
) result;

1 rows affected.

result
danke schön


:::{attention} Функция `ascii()`
:class: simple
:icon: false
Возвращает номер (эквивалент в ASCII) крайнего слева символа в переданной строке.
:::

In [38]:
%%sql
SELECT ASCII('ö');

1 rows affected.

ASCII('ö')
195


:::{note} Почему `ASCII('ö')` возвращает 195 вместо 148 (как в книге):

В современном UTF-8 (utf8mb4) буква `ö` состоит из двух байт: 195 и 182. Функция ASCII() считывает только самый первый байт строки, поэтому возвращает 195.

В книге автора использовалась старая однобайтовая кодировка CP850, где буква `ö` кодируется одним единственным байтом со значением 148.

**Решение для воссоздания книжного результата**: чтобы получить 148, нужно принудительно перевести символ в кодировку автора
```sql
SELECT ASCII(CONVERT('ö' USING cp850));
```
:::

In [39]:
%%sql
SELECT ASCII(CONVERT('ö' USING cp850)) AS book_result;

1 rows affected.

book_result
148


Используя функции `char()`, `ascii()` и `concat()` можно справиться с любой латиницей, даже если под рукой клавиатура без диакритических знаков и специальных символов.

---

### Манипуляции строками

Каждый сервер данных включает множество встроенных функций для манипуляции строками.

Прежде чем продолжить, сбросим и обновим данные в _string_tbl_:

In [43]:
%%sql
DELETE FROM string_tbl;

1 rows affected.

++
||
++
++

In [44]:
%%sql
INSERT INTO string_tbl (char_fld, vchar_fld, text_fld)
VALUES ('This string is 28 characters',
    'This string is 28 characters',
    'This string is 28 characters');

1 rows affected.

++
||
++
++

In [45]:
%%sql
SELECT * FROM string_tbl;

1 rows affected.

char_fld,vchar_fld,text_fld
This string is 28 characters,This string is 28 characters,This string is 28 characters



---

#### Строковые функции, возвращающие числовые значения

:::{attention} Функция `length()`
:class: simple
:icon: false
Возвращает количество символов в строке:
:::

In [46]:
%%sql
SELECT LENGTH(char_fld) char_length,
  LENGTH(vchar_fld) varchar_length,
  LENGTH(text_fld) text_length
FROM string_tbl;

1 rows affected.

char_length,varchar_length,text_length
28,28,28


Увидим одни и те же результаты для всех строковых функций независимо от типа стобца, в котором хранятся строки.

:::{attention} Функция `position()`
:class: simple
:icon: false
Находит местоположение подстроки внутри строки:
:::

In [47]:
%%sql
SELECT POSITION('characters' IN vchar_fld)
FROM string_tbl;

1 rows affected.

POSITION('characters' IN vchar_fld)
19


Если подстрока не может быть найдена, функция `position ()` возвращает значение 0.

:::{attention}
При работе с базами данных следует помнить, что первый символ строки находится в позиции 1.

Возвращаемое значение 0 указывает, что подстрока не может быть найдена, а не то, что подстрока является началом строки в которой выполняется поиск.
:::

Нестандартная функция `locate()` подобна функции `position()` но допускает необязательный третий параметр, используемый для указания начальной позиции поиска:

In [48]:
%%sql
SELECT LOCATE('is', vchar_fld, 5)
FROM string_tbl;

1 rows affected.

"LOCATE('is', vchar_fld, 5)"
13


:::{attention} Функция сравнения строк `strcmp()`
:class: simple
:icon: false
Принимает в качестве аргументов две строки и возвращает одно из значений:
- `-1`, если первая строка предшествует второй в порядке сортировки;
- ` 0`, если строки идентичны;
- ` 1`, если первая строка в порядке сортировки идет после второй.

**Нечувствительна** к регистру.
:::

In [49]:
%%sql
DELETE FROM string_tbl;

1 rows affected.

++
||
++
++

Сначала посмотрим порядок сортировки пяти строк используя запрос, а затем как строки сравниваются одна с другой, используя `strcmp()`.

In [50]:
%%sql
INSERT INTO string_tbl (vchar_fld)
VALUES ('abcd'),
       ('xyz'),
       ('QRSTUV'),
       ('qrstuv'),
       ('12345');

5 rows affected.

++
||
++
++

In [51]:
%%sql
SELECT vchar_fld
FROM string_tbl
ORDER BY vchar_fld;

5 rows affected.

vchar_fld
12345
abcd
QRSTUV
qrstuv
xyz


In [52]:
%%sql
SELECT STRCMP('12345', '12345') 12345_12345,
  STRCMP('abcd', 'xyz') abcd_xyz,
  STRCMP('abcd', 'QRSTUV') abcd_QRSTUV,
  STRCMP('qrstuv', 'QRSTUV') qrstuv_QRSTUV,
  STRCMP('12345', 'xyz') 12345_xyz,
  STRCMP('xyz', 'qrstuv') xyz_qrstuv;

1 rows affected.

12345_12345,abcd_xyz,abcd_QRSTUV,qrstuv_QRSTUV,12345_xyz,xyz_qrstuv
0,-1,-1,0,-1,1


Наряду с функцией `strcmp()` MySQL позволят использовать для сравнения строк в предложении **select** операторы `like` и `regexp`.

Такие сравнения будут давать значения 1 (истина) или 0 (ложь).

In [53]:
%%sql
SELECT name, name LIKE '%y' ends_in_y
FROM category;

16 rows affected.

name,ends_in_y
Action,0
Animation,0
Children,0
Classics,0
Comedy,1
Documentary,1
Drama,0
Family,1
Foreign,0
Games,0


In [55]:
%%sql
SELECT name, name REGEXP 'y$' ends_in_y
FROM category;

16 rows affected.

name,ends_in_y
Action,0
Animation,0
Children,0
Classics,0
Comedy,1
Documentary,1
Drama,0
Family,1
Foreign,0
Games,0



---

#### Строковые функции, возвращающие строки

В некоторых случаях требуется изменять существующие строки, извлекая часть строки или добавляя к ней дополнительный текст.

In [54]:
%%sql
DELETE FROM string_tbl;

5 rows affected.

++
||
++
++

In [55]:
%%sql
INSERT INTO string_tbl (text_fld)
VALUES ('This string was 29 characters');

1 rows affected.

++
||
++
++

:::{attention} Функция `concat()`
:class: simple
:icon: false
Полезна во многих ситуациях.
:::

В том числе когда нужно добавить к хранимой строке дополнительные символы:

In [39]:
%%sql
UPDATE string_tbl
SET text_fld = CONCAT(text_fld, ' but now it is longer');

1 rows affected.

++
||
++
++

In [40]:
%%sql
SELECT text_fld
FROM string_tbl;

1 rows affected.

text_fld
This string was 29 charactersbut now it is longer but now it is longer


Еще одно распространенное использование функции `concat()` – построение строки из отдельных фрагментов данных.

In [58]:
import pandas as pd
pd.set_option('display.max_colwidth', None)

print(f'Pandas ver. {pd.__version__}')
print('Ограничение на макс. ширину столбцов в 50 символов отключено.')

Pandas ver. 3.0.5
Ограничение на макс. ширину столбцов в 50 символов отключено.


In [59]:
%config SqlMagic.autopandas = True

In [60]:
%%sql
SELECT CONCAT(first_name, ' ', last_name,
    ' has been a customer since ',
    date(create_date)) cust_narrative
FROM customer;

599 rows affected.

,cust_narrative
0,MARY SMITH has been a customer since 2006-02-14
1,PATRICIA JOHNSON has been a customer since 2006-02-14
2,LINDA WILLIAMS has been a customer since 2006-02-14
3,BARBARA JONES has been a customer since 2006-02-14
4,ELIZABETH BROWN has been a customer since 2006-02-14
...,...
594,TERRENCE GUNDERSON has been a customer since 2006-02-14
595,ENRIQUE FORSYTHE has been a customer since 2006-02-14
596,FREDDIE DUGGAN has been a customer since 2006-02-14
597,WADE DELVALLE has been a customer since 2006-02-14


Функция `concat()` может обрабатывать любое выражение, возвращающее строку, и даже преобразовывать числа и даты в строковый формат, о чем свидетельствует столбец даты (create_date), использованный в качестве аргумента.

---

In [61]:
%config SqlMagic.autopandas = False

Хотя функция `concat()` полезна для добавления символов в начало или конец строки, может потребоваться добавить или заменить символы в середине строки.

:::{attention} Функция `insert()`
:class: simple
:icon: false
Принимает четыре аргумента:
- исходную строку,
- позицию, с которой следует начинать вставку,
- количество вставляемых символов,
- вставляемую строку.

В зависимости от значения третьего аргумента функция может использоваться как для вставки, так и для замены символов в строке. \
При значении третьего аргумента `0` строка вставляется (любые завершающие символы сдвигаются вправо):

In [62]:
%%sql
SELECT INSERT('goodbye world', 9, 0, 'cruel ') string;

1 rows affected.

string
goodbye cruel world


Если третий аргумент больше нуля, то он указывает количество символов, которые заменяются вставляемой строкой:

In [63]:
%%sql
SELECT INSERT('goodbye world', 1, 7, 'hello') string;

1 rows affected.

string
hello world


:::{attention} Функция `replace()`
:class: simple
:icon: false
Предназначена для замены одной подстроки другой.

Заменяет _каждый_ экземпляр искомой подстроки заменяемой строкой, поэтому нужно быть осторожным, чтобы не получить больше замен, чем ожидается.
:::

In [64]:
%%sql
SELECT REPLACE('goodbye world goodbye', 'goodbye', 'hello') string;

1 rows affected.

string
hello world hello


Помимо вставки символов в строку, может потребоваться извлечение подстроки из строки.

:::{attention} Функция `substring()`
:class: simple
:icon: false
Извлекает указанное количество символов, начиная с заданной позиции.
:::

In [65]:
%%sql
SELECT SUBSTRING('goodbye cruel world', 9, 5);

1 rows affected.

"SUBSTRING('goodbye cruel world', 9, 5)"
cruel



---

## Работа с числовыми данными

В отличие от строковых и временн*ы*х данных, генерация числовых данных довольно проста. Можете ввести число, получить его из другого столбца или сгенерировать его путем вычислений. При выполнении вычислений доступны все обычные арифметические операторы, а для изменения приоритетов вычислений можно использовать скобки.

In [66]:
%%sql
SELECT (37 * 59) / (78 - (8 * 6));

1 rows affected.

(37 * 59) / (78 - (8 * 6))
72.7667


Основная проблема при хранении числовых данных заключается в том, что числа могут быть округлены, если они больше размера, указанного для числового столбца. Например, число 9,96 будет округлено до 10,0 если оно сохранено в столбце определенном как `float(3,1)`.

### Выполнение математических функций

:::{note} Числовые функции от одного аргумента
:class: simple
:open: true
:icon: false

```text
| Имя функции | Описание                         |
| ----------- | -------------------------------- |
| acos(х)     | Вычисляет арккосинус х           |
| asin(х)     | Вычисляет арксинус х             |
| atan(х)     | Вычисляет арктангенс х           |
| cos(х)      | Вычисляет косинус х              |
| cot(х)      | Вычисляет котангенс х            |
| ехр(х)      | Вычисляет экспоненту х           |
| ln(x)       | Вычисляет натуральный логарифм х |
| sin(х)      | Вычисляет синус х                |
| sqrt(х)     | Вычисляет квадратный корень х    |
| tan(х)      | Вычисляет тангенс х              |
```
:::

:::{attention} Функция `mod()`
:class: simple
:icon: false
Оператор вычисления остатка от деления одного числа на другое.

В MySQL работает не только с целочисленными аргументами, но и с действительными числами.
:::

In [67]:
%%sql
SELECT MOD(10, 4);

1 rows affected.

"MOD(10, 4)"
2


In [68]:
%%sql
SELECT MOD(22.75, 5);

1 rows affected.

"MOD(22.75, 5)"
2.75


:::{attention} Функция `pow()`
:class: simple
:icon: false
Возвращает одно число возведенное в степень, равную значению второго числа.
:::

In [69]:
%%sql
SELECT POW(2, 8);

1 rows affected.

"POW(2, 8)"
256.0


In [70]:
%%sql
SELECT POW(2, 10) kilobyte, POW(2, 20) megabyte,
  POW(2, 30) gigabyte, POW(2, 40) terabyte;

1 rows affected.

kilobyte,megabyte,gigabyte,terabyte
1024.0,1048576.0,1073741824.0,1099511627776.0


### Управление точностью чисел

При работе с числами с плавающей точкой не всегда требуется работа или отображение числа с полной точностью. Например, можно хранить данежные данные транзакции с точностью до шести знаков после запятой, но для их отображения понадобится округление до ближайшей сотой.

:::{attention} Функции `ceil()` и `floor()`
:class: simple
:icon: false
Используются для округления в б*о*льшую или меньшую сторону к ближайшему целому.
:::

In [82]:
%%sql
SELECT CEIL(72.445), FLOOR(72.445);

1 rows affected.

CEIL(72.445),FLOOR(72.445)
73,72


In [81]:
%%sql
SELECT CEIL(72.000000001), FLOOR(72.999999999);

1 rows affected.

CEIL(72.000000001),FLOOR(72.999999999)
73,72


:::{attention} Функция `round()`
:class: simple
:icon: false
Округляет к ближайшему целому по обычным правилам.
:::

In [87]:
%%sql
SELECT ROUND(72.49999), ROUND(72.5), ROUND(72.50001);

1 rows affected.

ROUND(72.49999),ROUND(72.5),ROUND(72.50001)
72,73,73


В большинстве случаев требуется сохранить хотя бы некоторую часть десятичных знаков после запятой, а не округлять значение до целого числа.

Для этого функция `round()` допускает необязательный второй аргумент, который указывает какое количество цифр справа от запятой следует оставить при округлении.

In [88]:
%%sql
SELECT ROUND(72.0909, 1), ROUND(72.0909, 2), ROUND(72.0909, 3);

1 rows affected.

"ROUND(72.0909, 1)","ROUND(72.0909, 2)","ROUND(72.0909, 3)"
72.1,72.09,72.091


:::{attention} Функция `truncate()`
:class: simple
:icon: false
Отбрасывает _(усекает)_ ненужные цифры без округления.

Обязательный второй аргумент указывает количество цифр справа от десятичной запятой.
:::

In [90]:
%%sql
SELECT TRUNCATE(72.000000001, 0), TRUNCATE(72.999999999, 0);

1 rows affected.

"TRUNCATE(72.000000001, 0)","TRUNCATE(72.999999999, 0)"
72,72


In [91]:
%%sql
SELECT TRUNCATE(72.0909, 1), TRUNCATE(72.0909, 2), TRUNCATE(72.0909, 3);

1 rows affected.

"TRUNCATE(72.0909, 1)","TRUNCATE(72.0909, 2)","TRUNCATE(72.0909, 3)"
72.0,72.09,72.090


:::{note} Примечание
И `truncate()` и `round()` допускают *отрицательные* значения второго аргумента. Это означает, что усекаются или округляются цифры *слева* от десятичной точки.

Это поначалу может показаться странным, но для таких значений аргумента имеются вполне допустимые ситуации. Например, вы можете продавать продукт, который можно покупать только по 10 единиц. Если покупателем заказаны 17 единиц, можете выбрать один из спосовоб изменения количества товара в заказе клиента:
:::

In [94]:
%%sql
SELECT ROUND(17, -1), TRUNCATE(17, -1);

1 rows affected.

"ROUND(17, -1)","TRUNCATE(17, -1)"
20,10


### Работа со знаковыми данными

Если работаете с числовыми столбцами, в которых допускаются отрицательные значения, могут быть полезны некоторые специализированные числовые функции.

Допустим вас просят создать отчет, показывающий текущее состояние набора банковских счетов, используя данные из некой таблицы account:
```text
| account_id | acct_type    | balance |
| ---------- | ------------ | ------- |
| 123        | MONEY MARKET | 785.22  |
| 456        | SAVINGS      | 0.00    |
| 789        | CHECKING     | -324.22 |
```

Представленный далее запрос возвращает три столбца, полезных при создании отчета:
```sql
SELECT account_id, SIGN(balance), ABS(balance)
FROM account;
```
```text
| account_id | SIGN(balance) | ABS(balance) |
| ---------- | ------------- | ------------ |
| 123        | 1             | 785.22       |
| 456        | 0             | 0.00         |
| 789        | -1            | 324.22       |
```
Во втором столбце функция `sign()` возвращает значение -1, если баланс счета отрицательный, 0 если баланс нулевой и 1 если баланс положительный. Третий столбец получает абсолютное значение баланса счета с помощью функции `abc()`.

---

## Работа с временн*ы*ми данными

Из трех типов данных, обсуждаемых в этой главе (символьные, числовые и временн*ы*е), когда дело доходит до создания и обработки данных, временн*ы*е данные являются наиболее сложными.

Некоторая сложность вызвана множеством способов, которыми можно описать единственную дату и время. Б*о*льшая часть сложности связана с системой отсчета.

### Часовые пояса

:::{note} GMT & UTC

- **GMT** (Greenwich Mean Time) – среднее время по Гринвичу (устаревшее)
- **UTC** (Coordinated Universal Time) – всемирное координированное время (современное)

И SQL Server и MySQL предоставляют функции, которые будут возвращать текущую метку времени UTC: `getutcdate()` – для SQL Server и `utc_timestamp()` – для MySQL.
:::

MySQL хранит два разных часовых пояса: **глобальный** часовой пояс и часовой пояс **сеанса**, которые могут быть разными для каждого пользователя вошедшего в базу данных.

In [2]:
%%sql
SELECT @@global.time_zone, @@session.time_zone;

1 rows affected.

@@global.time_zone,@@session.time_zone
SYSTEM,SYSTEM


Значение **system** говорит о том, что сервер использует настройку часового пояса, в котором находится база данных.

Если вы сидите за компьютером в Санкт-Петербурге (Россия) и открываете сеанс по сети на сервере MySQL расположенном в Нью-Йорке, то можете изменить настройку часового пояса для своего сеанса с помощью команды:

In [7]:
%%sql
SET time_zone = 'Europe/Moscow';

RuntimeError: (pymysql.err.OperationalError) (1298, "Unknown or incorrect time zone: 'Europe/Moscow'")
[SQL: SET time_zone = 'Europe/Moscow';]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


:::{note} Примечание
После установки на локальном компе MySQL знает только системное время компьютера (system). Буквенные названия городов ('Europe/Zurich', 'Europe/Moscow') он распознавать не умеет, пока администратор базы данных не выполнит специальную команду в консоли операционной системы (не загрузит таблицы временных зон).

В таком случае можно указать смещение в часах:
:::

In [5]:
%%sql
SET time_zone = '+03:00';

++
||
++
++

In [6]:
%%sql
SELECT @@global.time_zone, @@session.time_zone;

1 rows affected.

@@global.time_zone,@@session.time_zone
SYSTEM,+03:00


Чтобы сеанс **снова** синхронизировался с системным временем самого сервера *(а в качестве сервера мы используем наш собственный локальный компьютер):*

In [8]:
%%sql
SET time_zone = 'SYSTEM';

++
||
++
++

In [9]:
%%sql
SELECT @@global.time_zone, @@session.time_zone;

1 rows affected.

@@global.time_zone,@@session.time_zone
SYSTEM,SYSTEM



---

### Генерация временн*ы*х данных

Генерировать временн*ы*е данные можно любым из способов:
- копирование данных из существующего столбца `date`, `datetime` или `time`;
- выполнение встроенной функции, которая возвращает `date`, `datetime` или `time`;
- построение строкового представления временн*ы*х данных для вычисления сервером.

Чтобы использовать последний метод, необходимо знать о различных компонентах, используемых при формировании дат.

#### Строковые представления временн*ы*х данных

:::{attention} Компоненты формата даты
:class: simple
:icon: false

| Компонент | Определение      | Диапазон                       |
| --------- | ---------------- | ------------------------------ |
| YYYY      | Год, включая век | От 1000 до 9999                |
| ММ        | Месяц            | От 01 (январь) до 12 (декабрь) |
| DD        | День             | От 01 до 31                    |
| HH        | Час              | От 00 до 23                    |
| HHH       | Часы (прошедшие) | От -838 до 838                 |
| MI        | Минуты           | От 00 до 59                    |
| SS        | Секунды          | От 00 до 59                    |
:::

:::{attention} Компоненты дат и времени
:class: simple
:icon: false

Чтобы создать строку, которую сервер может интерпретировать как `date`, `datetime` или `time`, необходимо собрать вместе различные компоненты в порядке, указанном в таблице:

| Тип       | Формат              | Допустимые значения                                             |
| --------- | ------------------- | --------------------------------------------------------------- |
| date      | YYYY-MM-DD          | От 1000-01-01 до 9999-12-31                                     |
| datetime  | YYYY-MM-DD HH:MI:SS | От 1000-01-01 00:00:00.000000 <br>до 9999-12-31 23:59:59.999999 |
| timestamp | YYYY-MM-DD HH:MI:SS | От 1970-01-01 00:00:00.000000<br>до 2038-01-18 22:14:07.999999  |
| year      | YYYY                | От 1901 до 2155                                                 |
| time      | HHH:MI:SS           | От -838:59:59.000000 до 838:59:59.000000                        |

То есть, например, чтобы заполнить столбец `datetime` значением 15:30 17 сентября 2019 года, нужно постороить строку `2019-09-17 15:30:00`
:::

Пример инструкции, используемой для изменения даты возврата взятого напрокат фильма:
```sql
UPDATE rental
SET return_date = '2019-09-17 15:30:00'
WHERE rental_id = 99999;
```
Сервер определяет, что строка, указанная в предложении `set` должна быть значением `datetime` так как строка используется для заполнения столбца `datetime`. Поэтому сервер попытается преобразовать строку, разделяя ее на шесть компонентов (год, месяц, день, час, минута, секунда), включаемых в формат `datetime` по умолчанию.

---

#### Преобразование строки в дату

Если сервер *не* ожидает значения `datetime` или если вы хотите представить `datetime` в формате, отличном от формата по умолчанию, нужно указать серверу на необходимость преобразования строки в дату и время.

:::{attention} Функция `cast()`
:class: simple
:icon: false
Используется для явного приведения значения из одного типа данных в другой.
```sql
CAST(выражение AS тип_данных)
```
- DATE – превращает строку в чистую дату (2026-08-25)
- DATETIME – превращает строку в дату со временем
- TIME – превращает строку во временной интервал
:::

Пример простого запроса, который возвращает значение `datetime` c помощью функции `cast()`:

In [10]:
%%sql
SELECT CAST('2019-09-17 15:30:00' AS DATETIME);

1 rows affected.

CAST('2019-09-17 15:30:00' AS DATETIME)
2019-09-17 15:30:00


Та же логика применяется и к типам `date` и `time`:

In [13]:
%%sql
SELECT CAST('2019-09-17' AS DATE) date_field,
CAST('108:17:57' AS TIME) time_field;

1 rows affected.

date_field,time_field
2019-09-17,"4 days, 12:17:57"


:::{note} Отображение больших значений `TIME` в Python/Jupyter

Тип данных `TIME` в MySQL может хранить длительность до 838 часов. Консоль автора (CLI) выводит такие значения *как есть* (108:17:57).

Однако встроенный тип времени в Python ограничен 24 часами. Поэтому драйвер PyMySQL автоматически преобразует любые значения времени больше суток в объект интервала `timedelta`, который Jupyter выводит на экран в формате дней и часов: 4 days, 12:17:57.

---
Чтобы отключить это поведение драйвера и увидеть оригинальный формат из книги, значение нужно принудительно перевести в текстовую строку на стороне СУБД:
```sql
SELECT CAST(CAST('108:17:57' AS TIME) AS CHAR);
```
:::

In [12]:
%%sql
SELECT CAST(CAST('108:17:57' AS TIME) AS CHAR) AS time_field;

1 rows affected.

time_field
108:17:57


:::{important}
Вместо того, чтобы позволить серверу выполнять неявные преобразования – рекомендуется выполнять их **явно**, даже если сервер ожидает значение `date`, `datetime` или `time`.
:::

In [27]:
%%sql
SELECT * FROM rental 
WHERE rental_date < CAST('2005-05-25' AS DATE);

8 rows affected.

rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
1,2005-05-24 22:53:30,367,130,2005-05-26 22:04:30,1,2006-02-15 21:30:53
2,2005-05-24 22:54:33,1525,459,2005-05-28 19:40:33,1,2006-02-15 21:30:53
3,2005-05-24 23:03:39,1711,408,2005-06-01 22:12:39,1,2006-02-15 21:30:53
4,2005-05-24 23:04:41,2452,333,2005-06-03 01:43:41,2,2006-02-15 21:30:53
5,2005-05-24 23:05:21,2079,222,2005-06-02 04:33:21,1,2006-02-15 21:30:53
6,2005-05-24 23:08:07,2792,549,2005-05-27 01:32:07,1,2006-02-15 21:30:53
7,2005-05-24 23:11:53,3995,269,2005-05-29 20:34:53,2,2006-02-15 21:30:53
8,2005-05-24 23:31:46,2346,239,2005-05-27 23:33:46,2,2006-02-15 21:30:53


Когда строки преобразуются во временные значения (явно или неявно) вы должны предоставить все компоненты даты в необходимом порядке.

---

#### Функции генерации дат

Если нужно сгенерировать временн*ы*е данные из строки, а строка находится в неверном виде для использования функции `cast()`, можно использовать встроенную функцию, которая принимает вместе со строкой даты строку формата.

:::{attention} Функция `str_to_date()`
:class: simple
:icon: false

Используется для **явного** приведения сложных или нестандартных строк к типу данных даты/времени.
:::

Пусть, например, вы извлекаете из файла строку 'September 17, 2019' и вам нужно использовать ее для обновления столбца `date`. Поскольку строка не соответствует требуемому формату YYYY-MM-DD, можем использовать `str_to_date()` вместо того, чтобы переформатировать строку для использования функции `cast()`:

```sql
UPDATE rental
SET return_date = STR_TO_DATE('September 17, 2019', '%M', '%d', '%Y')
WHERE rental_id = 99999;
```

Второй аргумент в вызове `str_to_date()` определяет формат строки даты. В данном случае строка содержит название месяца `%M`, числовое значение дня `%d` и четырехзначное числовое знаение года `%Y`.

:::{attention} Компоненты формата даты
:class: simple
:icon: false
Хотя имеется более 30 распознаваемых компонентов формата, в этой таблице показаны полтора десятка наиболее часто используемых:

| Компонент формата | Описание                                              |
| ----------------- | ----------------------------------------------------- |
| %М                | Полное имя месяца (January..December)                 |
| %m                | Числовое значение месяца (01, 02...12)                |
| %c                | Числовое значение месяца (1, 2...12)                  |
| %d                | Числовое значение дня месяца (00... 31)               |
| %j                | День года (001.. 366)                                 |
| %W                | Полное имя дня недели (Sunday... Saturday)            |
| %w                | Числовое значение дня недели (0=Sunday... 6=Saturday) |
| %Y                | Значение года (четыре цифры)                          |
| %y                | Значение года (две цифры)                             |
| %Н                | Час дня в 24-часовом формате (00... 23)               |
| %h                | Час дня в 12-часовом формате (01... 12)               |
| %i                | Минуты в часе (00... 59)                              |
| %s                | Число секунд (00... 59)                               |
| %f                | Число микросекунд (000000... 999999)                  |
| %р                | AM или РМ                                             |
| %а                | Краткое имя дня недели (Sun, Mon, ...)                |
| %b                | Краткое имя месяца (Jan, Feb, ...)                    |
:::

Функция `str_to_date()` возвращает значение `datetime`, `date` или `time` в зависимости от содержимого строки формата. Например, если строка формата включает только %H, %i и %s, будет возвращено значение `time`.

Если пытаетесь сгенерировать *текущую* дату/время, то не нужно создавать строку, потому что имеются встроенные функции, которые обращаются к системным часам и возвращают текущую дату и/или время в виде строки:

In [28]:
%%sql
SELECT CURRENT_DATE(), CURRENT_TIME(), CURRENT_TIMESTAMP();

1 rows affected.

CURRENT_DATE(),CURRENT_TIME(),CURRENT_TIMESTAMP()
2026-08-25,1:57:11,2026-08-25 01:57:11


Значения, возвращаемые этими функциями, имеют формат по умолчанию для возвращаемого временного типа.

---

### Манипуляции временн*ы*ми данными

В этом разделе исследуются встроенные функции, которые принимают аргументы даты и возвращают даты, строки или числа.

#### Функции, возвращающие даты

Многие встроенные временн*ы*е функции принимают в качестве аргумента одну дату и возвращают другую.

:::{attention} Функция `date_add()`
:class: simple
:icon: false
Позволяет добавлять любые интервалы (например, дни, месяцы, годы) к указанной дате для создания другой даты.
:::

Пример, как добавить к текущей дате пять дней:

In [9]:
%%sql
SELECT DATE_ADD(CURRENT_DATE(), INTERVAL 5 DAY);

1 rows affected.

"DATE_ADD(CURRENT_DATE(), INTERVAL 5 DAY)"
2026-08-31


In [10]:
%%sql
SELECT DATE_ADD(CURRENT_DATE(), INTERVAL -364 DAY);

1 rows affected.

"DATE_ADD(CURRENT_DATE(), INTERVAL -364 DAY)"
2025-08-27


Второй аргумент состоит из трех элементов: ключевого слова `interval`, желаемого количества и типа интервала.

:::{attention} Распространенные типы интервалов
:class: simple
:icon: false

| Имя интервала | Описание                                          |
| ------------- | ------------------------------------------------- |
| second        | Количество секунд                                 |
| minute        | Количество минут                                  |
| hour          | Количество часов                                  |
| day           | Количество дней                                   |
| month         | Количество месяцев                                |
| year          | Количество лет                                    |
| minute_second | Количество минут и секунд, разделенных `:`        |
| hour_second   | Количество часов, минут и секунд, разделенных `:` |
| year_month    | Количество лет и месяцев, разделенных `-`         |

Первые шесть типов просты, последние три типа требуют пояснения, поскольку имеют по несколько элементов:
:::

Например, вам сказали, что фильм вернули на 3 часа 27 минут 11 секунд позже, чем было указано изначально. Чтобы исправить:

```sql
UPDATE rental
SET return_date = DATE_ADD(return_date, INTERVAL '3:27:11' HOUR_SECOND)
WHERE rental_id = 99999;
```
Или, например, в отделе кадров выяснил, что сотрудник с id 4789 в базе данных старше на 9 лет и 11 месяцев, чем на самом деле:

```sql
UPDATE employee
SET birth_date = DATE_ADD(birth_date, INTERVAL '9-11' YEAR_MONTH)
WHERE emp_id = 4789;
```

Например, клиент банка входит 17 сентября 2019 года в систему онлайн-банкинга и планирует перевод на конец месяца:

:::{attention} Функция `last_day()`
:class: simple
:icon: false
Принимает на вход дату или дату со временем и возвращает **последний день месяца** для указанного значения.
```sql
LAST_DAY(дата)
```

---
- Автоматически учитывает **високосные годы**
- Возвращает тип данных `DATE`
- Если передать некорректную дату (например, '2026-02-31'), функция вернет `NULL`
:::

In [9]:
%%sql
SELECT LAST_DAY('2019-09-17');

1 rows affected.

LAST_DAY('2019-09-17')
2019-09-30


Независимо от того, указываете ли вы значение `date` или `datetime`, функция `last_day()` всегда возвращает `date`.

---

#### Функции, возвращающие строки

Большинство временн*ы*х функций, возвращающих строковые значения, используются для извлечения части даты или времени.

:::{attention} Функция `dayname()`
:class: simple
:icon: false
Определяет на какой день недели выпадает определенная дата.
:::

In [10]:
%%sql
SELECT DAYNAME('2019-09-18');

1 rows affected.

DAYNAME('2019-09-18')
Wednesday


:::{card}
В MySQL много функций для извлечения информации из значений даты, но автор рекомендует вместо них использовать функцию `extract()`, так как проще запомнить несколько вариантов одной функции, чем десяток различных функций.
:::

:::{attention} Функция `extract()`
:class: simple
:icon: false
Используется для явного извлечения конкретного компонента (года, месяца, дня, часа и т.д.) из указанного значения даты или времени.

```sql
EXTRACT(компонент FROM дата_время)
```

---
- Возвращает результат в виде обычного целого числа (тип `INTEGER`)
- Поддерживает как простые компоненты (YEAR, MONTH, DAY, HOUR), так и сложные интервалы (YEAR_MONTH, DAY_MINUTE)
- Является частью международного стандарта ANSI SQL, поэтому код с `EXTRACT()` будет одинаково работать в MySQL, PostgreSQL и Oracle.
:::

Например, чтобы извлечь из значения `datetime` только часть года:

In [12]:
%%sql
SELECT EXTRACT(YEAR FROM '2019-09-18 22:19:05');

1 rows affected.

EXTRACT(YEAR FROM '2019-09-18 22:19:05')
2019



---

#### Функции, возвращающие числовые значения

Еще одна распространенная задача при работе с датами – определение количества интервалов (дней, недель, лет) между двумя датами.

:::{attention} Функция `datediff()`
:class: simple
:icon: false

Возвращает количество полных дней между двумя датами.

```sql
DATEDIFF(конечная_дата, начальная_дата)
```

---
Всегда вычитает второе значение из первого (`конечная_дата` минус `начальная_дата`).
- Если конечная дата в будущем – результат будет положительным (столько дней нужно добавить).
- Если конечная дата в прошлом – результат будет отрицательным.
:::

In [3]:
%%sql
SELECT DATEDIFF('2019-09-03', '2019-06-21');

1 rows affected.

"DATEDIFF('2019-09-03', '2019-06-21')"
74


Функция `datediff()` игнорирует время дня в своих аргументах:

In [4]:
%%sql
SELECT DATEDIFF('2019-09-03 23:59:59', '2019-06-21 00:00:01');

1 rows affected.

"DATEDIFF('2019-09-03 23:59:59', '2019-06-21 00:00:01')"
74


Если поменять аргументы местами и сначала указать более раннюю дату, `datediff()` вернет отрицательное значение:

In [5]:
%%sql
SELECT DATEDIFF('2019-06-21', '2019-09-03');

1 rows affected.

"DATEDIFF('2019-06-21', '2019-09-03')"
-74


Количество недель или лет можем высчитать простым делением на 7 дней в неделю или 365 дней в году:

In [22]:
%%sql
SELECT DATEDIFF('2026-08-26', '2025-08-27') / 365;

1 rows affected.

"DATEDIFF('2026-08-26', '2025-08-27') / 365"
0.9973


Если нужна разница не в днях, а в других единицах, вместо `datediff()` используют функцию `timestampdiff()`. Она умеет считать и недели, и года, и месяцы.

::::{attention} Функция `timestampdiff()`
:class: simple
:icon: false

Вычисляет разницу между двумя временными метками в заданных единицах измерения (годах, месяцах, неделях, днях, часах и т.д.).

```sql
TIMESTAMPDIFF(единица_времени, начальная_дата, конечная_дата)
```

---
- В отличие от `datediff()` функция `timestampdiff()` вычитает первую дату из второй;
- Допустимые единицы времени: YEAR (года), QUARTER (кварталы), MONTH (месяцы), WEEK (недели), DAY (дни), HOUR (часы), MINUTE (минуты), SECOND (секунды)
- **Точность расчета**: возвращает *полное* количество периодов, автоматически округляя результат в меньшую сторону (вниз) по фактическому заполнению интервала.

:::{div}
:class: text-sm
Например, если между датами прошло 11 месяцев и 29 дней, `TIMESTAMPDIFF(YEAR, ...)` вернет `0`, а `TIMESTAMPDIFF(MONTH, ...)` вернет `11`.
:::
::::

In [62]:
%%sql
SELECT 
    TIMESTAMPDIFF(DAY, '2025-08-27', '2026-08-26') AS days_diff,
    TIMESTAMPDIFF(MONTH, '2025-08-27', '2026-08-26') AS months_diff,
    TIMESTAMPDIFF(YEAR, '2025-08-27', '2026-08-26') AS years_diff
;

1 rows affected.

days_diff,months_diff,years_diff
364,11,0



---

## Функции преобразования

[Ранее было показано](https://magus1968.github.io/learning-sql/ch07/#id-15) как использовать функцию `cust()` для преобразования строки в значение `datetime`.

Чтобы использовать `cast()`, вы предоставляете значение или выражение, ключевое слово `as` и тип, в который хотите преобразовать это значение.

Вот пример преобразования строки в целое число:

In [23]:
%%sql
SELECT CAST('1456328' AS SIGNED INTEGER);

1 rows affected.

CAST('1456328' AS SIGNED INTEGER)
1456328


При преобразовании строки в число функция `cast()` пытается преобразовать всю строку слева направо. Если в строке обнаруживается знак, которого не может быть в числе, преобразование останавливается без сообщения об ошибке:

In [24]:
%%sql
SELECT CAST('999ABC111' AS UNSIGNED INTEGER);

1 rows affected.

CAST('999ABC111' AS UNSIGNED INTEGER)
999


Преобразуются первые три цифры строки, а остальные отбрасываются. При этом сервер `MySQL CLI` выдает предупреждение, чтобы вы знали, что не вся строка была преобразована:
```text
+---------------------------------------+
| CAST('999ABC111' AS UNSIGNED INTEGER) |
+---------------------------------------+
|                                   999 |
+---------------------------------------+
1 row in set, 1 warning (0.00 sec)
```
:::{card}
К сожалению не нашел способ как реализовать появление предупреждения `1 warning` в Jupyter.
:::

In [25]:
%%sql
SHOW WARNINGS;

1 rows affected.

Level,Code,Message
Warning,1292,Truncated incorrect INTEGER value: '999ABC111'


Если вы конвертируете строку в значение `date`, `datetime` или `time`, следует придерживаться форматов по умолчанию для каждого типа, поскольку предоставить функции `cast()` строку формата нельзя. `cast()` умеет переводить строку в дату **только в том случае, если строка уже записана в эталонном формате ISO**.

Если ваша строка даты представлена *не* в формате по умолчанию (т.е. `YYYY-MM-DD HH:MI:SS` для типа `datetime`), то нужно использовать другую [рассмотренную ранее](https://magus1968.github.io/learning-sql/ch07/#id-16) функцию `str_to_date()`

---

## Упражнения

### Упражнение 7.1

Напишите запрос, который возвращает символы строки 'Please find the substring in this string' с 17-го по 25-й.

In [42]:
%%sql
SELECT SUBSTRING('Please find the substring in this string', 17, 9);

1 rows affected.

"SUBSTRING('Please find the substring in this string', 17, 9)"
substring



---

### Упражнение 7.2

Напишите запрос, который возвращает абсолютное значение и знак (-1, 0 или 1) числа -25,76823. Верните также число, округленное до ближайших двух знаков после запятой.

In [46]:
%%sql
SELECT ABS(-25.76823), SIGN(-25.76823), ROUND(-25.76823, 2);

1 rows affected.

ABS(-25.76823),SIGN(-25.76823),"ROUND(-25.76823, 2)"
25.76823,-1,-25.77



---

### Упражнение 7.3

Напишите запрос, возвращающий для текущей даты только часть, соответствующую месяцу.

In [47]:
%%sql
SELECT EXTRACT(MONTH FROM CURRENT_DATE());

1 rows affected.

EXTRACT(MONTH FROM CURRENT_DATE())
8


:::{div}
:class: framed text-center
Задача решена, но стало интересно: \
как получить название месяца, причем на русском языке?
:::

:::{attention} Функция `monthname()`
:class: simple
:icon: false
Принимает дату и возвращает полное название месяца в виде строки
```sql
MONTHNAME(дата)
```
:::

In [50]:
%%sql
SELECT 
    MONTHNAME(CURRENT_DATE()) current_month_name;

1 rows affected.

current_month_name
August


Если в будущем потребуется вывести не просто имя месяца, а, например, сокращенное название (Aug) или совместить его с годом (August 2026), лучшим выбором станет `date_format()`.

:::{attention} Функция `date_format()`
:class: simple
:icon: false

Преобразовывает объекты даты и времени в строку по заданному шаблону (маске)
```sql
DATE_FORMAT(дата_или_время, '%компонент_формата')
```

---
Все символы, перед которыми нет знака `%` (дефисы, пробелы, точки, запятые, цифры или целые слова), выводятся в итоговую строку *как есть*.
:::

In [59]:
%%sql
SELECT
    DATE_FORMAT(CURRENT_DATE(), '%M') AS month_name_universal,
    DATE_FORMAT(CURRENT_DATE(), '%b') AS month_name_short,
    DATE_FORMAT(CURRENT_DATE(), '%d.%m.%Y') AS russian_date_format
;

1 rows affected.

month_name_universal,month_name_short,russian_date_format
August,Aug,26.08.2026


:::{attention} Язык вывода текстовых значений
:class: simple
:icon: false

(названий месяцев и дней недели) напрямую зависит от серверной переменной `lc_time_names`. По умолчанию в MySQL установлена локаль `en_US`. Чтобы переключить вывод на русский язык, перед основным запросом необходимо выполнить команду смены локали.

```sql
SET lc_time_names = 'ru_RU';
```
:::

In [60]:
%%sql
-- 1. Переключаем локаль сессии на русскую (для текущего запроса)
SET lc_time_names = 'ru_RU';

-- 2. Выполняем целевой запрос
SELECT
    DATE_FORMAT(CURRENT_DATE(), '%W, %d %M') AS formatted_date,
    DATE_FORMAT(NOW(), '%W, %d %M %Y года, %H:%i') AS full_human_datetime
;

1 rows affected.

formatted_date,full_human_datetime
"Среда, 26 Августа","Среда, 26 Августа 2026 года, 04:44"


:::{note} `CURRENT_DATE()` vs `NOW()`
:class: simple
:icon: false
- `CURRENT_DATE()` возвращает только дату в формате `YYYY-MM-DD`. В ней принципиально нет информации о часах, минутах и секундах. Если передать её в `DATE_FORMAT()` вместе со спецификаторами времени `%H:%i`, функция вернет нули `00:00`, так как данным о времени просто взяться не откуда.
- `NOW()` возвращает и дату, и текущее время в формате `YYYY-MM-DD HH:MM:SS`. В маске второго запроса появились спецификаторы времени `%H:%i`, тип данных пришлось повысить до `NOW()`, чтобы получить реальное текущее время сервера.
:::

:::{important} Важно 
Когда выполняем ячейку с `%%sql`, расширение JupySQL не закрывает соединение с базой данных сразу после вывода таблицы. Оно удерживает одну активную сессию (подключение) в рамках всего текущего сеанса работы ноутбука.

То есть измененная (русская) локаль останется активной для всех последующих ячеек, пока не перезапустим ядро (Kernel) ноутбука или не закроем соединение.

---
Правильный подход – локально изменить настройку, забрать данные и сразу **вернуть всё в исходное (дефолтное) состояние**.
:::

In [61]:
%%sql
-- 3. Возвращаем локаль сессии в стандартный режим
SET lc_time_names = 'en_US';

-- 4. Проверяем возврат дефолтных настроек
SELECT
    DATE_FORMAT(CURRENT_DATE(), '%W, %d %M') AS formatted_date;

1 rows affected.

formatted_date
"Wednesday, 26 August"



---